# Resource estimation for SOSSA sum-of-squares spectral amplification

This notebook estimates the cost of running **unary-iteration quantum phase estimation** on a
Hamiltonian block-encoded with **SOSSA** (sum-of-squares spectral amplification, Low *et al.*,
[arXiv:2502.15882](https://arxiv.org/abs/2502.15882)).

SOSSA writes an electronic-structure Hamiltonian as a shifted sum of squares,

$$
H \;=\; \underbrace{\sum_{x} c_x\, A_x^{\dagger} A_x}_{H_{\mathrm{gap}} \;\succeq\; 0} \;+\; E_{\mathrm{SOS}},
$$

where every generator $A_x$ is a Majorana bilinear drawn from a double-factorized (DFTHC)
representation of the two-body integrals. Because $H_{\mathrm{gap}}$ is positive semi-definite,
its ground state sits at the *edge* of the walk operator's spectral band, and the phase-to-energy
map

$$
E_{\mathrm{gap}}(\varphi) \;=\; 2\Lambda \cos^{2}(\pi\varphi)
$$

is quadratically flat there. That flatness is the amplification: a phase resolved to
$\delta\varphi \sim 1/T$ yields an energy resolved to $\delta E \sim 1/T^{2}$, so the query count
needed for a target precision scales far better than for a plain qubitization walk.

We run the full pipeline twice:

1. **Part 1 --- a stored DFTHC instance.** A small H<sub>2</sub> Hamiltonian loaded straight from
   JSON, already in the factorized form SOSSA consumes, with a non-zero `energy_gap` so the
   amplified walk is exercised.
2. **Part 2 --- stretched N<sub>2</sub>, built from scratch.** SCF, orbital localization,
   automated active-space selection, then a **double factorization** of the active-space
   two-body integrals to produce the SOSSA input.

Both parts end in the same place: a unary-iteration QPE circuit, its logical gate counts, and a
physical resource estimate from `qdk.qre`.

In addition to [installing `qdk-chemistry`](https://github.com/microsoft/qdk-chemistry/blob/main/INSTALL.md),
you will need the `qre`, `jupyter` and `qiskit-extras` extras:

```bash
pip install 'qdk-chemistry[jupyter,qiskit-extras,qre]'
```

In [ ]:
import math
from pathlib import Path

import pandas as pd

from qdk_chemistry.algorithms import create
from qdk_chemistry.data import (
    AlgorithmRef,
    Configuration,
    Hamiltonian,
    MajoranaMapping,
    StateVectorContainer,
    Wavefunction,
)
from qdk_chemistry.utils import Logger

# Reduce logging output for the demo
Logger.set_global_level(Logger.LogLevel.off)

## The shared pipeline

Both halves of the notebook funnel into the same three steps, so we define them once.

`SOSSAQubitMapper` turns a `FactorizedHamiltonianContainer` into a structured qubit operator that
records the generator coefficients and scalar shift $E_{\mathrm{SOS}}$. `SOSSABuilder` derives the
generator one-norms and block-encoding normalization $\Lambda$, then the unary-iteration QPE builder
consumes the operator directly: it allocates one phase qubit per query and reuses a single controlled
walk across every slot.

The circuit-mapper settings choose how each piece of the walk is synthesized:

- `outer_prepare` --- prepares $\sqrt{c_x}$-weighted amplitudes over the generator index. Alias
  sampling gives a gate count that grows linearly rather than exponentially in the register width.
- `inner_prepare_algorithm` --- the controlled rotation cascade that builds each Majorana bilinear.
- `select_algorithm` --- `qrom_phase_gradient` applies the basis rotations through a QROM lookup
  into a shared phase-gradient register, which trades rotation synthesis for Toffolis.
- `rotation_bit_precision` / `coefficient_bit_precision` --- how finely the Givens angles and the
  amplitudes are discretized. These dominate the T/Toffoli count, so they are the main knobs to
  sweep when tightening a resource estimate.

In [ ]:
def sossa_unary_qpe_circuit(
    hamiltonian,
    *,
    num_queries,
    n_alpha,
    n_beta,
    circuit_mapper=None,
    rotation_bit_precision=15,
    coefficient_bit_precision=11,
):
    """Build a unary-iteration QPE circuit driven by the SOSSA walk.

    Args:
        hamiltonian: A Hamiltonian backed by a FactorizedHamiltonianContainer.
        num_queries: Number of walk applications in the phase-estimation schedule.
        n_alpha: Number of alpha electrons in the reference determinant.
        n_beta: Number of beta electrons in the reference determinant.
        circuit_mapper: Optional SOSSA circuit mapper; defaults to the resource-estimation configuration.
        rotation_bit_precision: Bits used to discretize the Givens rotation angles.
        coefficient_bit_precision: Bits used to discretize the PREPARE amplitudes.

    Returns:
        A tuple of (QPE circuit, SOSSA qubit operator).

    """
    container = hamiltonian.get_container()
    num_orbitals = container.get_num_orbitals()
    orbitals = container.get_orbitals()

    operator = create("qubit_mapper", "sossa").run(
        hamiltonian, MajoranaMapping.jordan_wigner(2 * num_orbitals)
    )

    # Hartree-Fock reference determinant, loaded with the sparse isometry method.
    hf_config = Configuration.canonical_hf_configuration(n_alpha, n_beta, num_orbitals)
    reference = Wavefunction(StateVectorContainer(hf_config, orbitals))
    state_prep = create("state_prep", "sparse_isometry").run(reference)

    if circuit_mapper is None:
        circuit_mapper = AlgorithmRef(
            "circuit_mapper",
            "sossa",
            outer_prepare=AlgorithmRef("state_prep", "alias_sampling"),
            inner_prepare_algorithm="controlled_alias_sampling",
            select_algorithm="qrom_phase_gradient",
            rotation_bit_precision=rotation_bit_precision,
            coefficient_bit_precision=coefficient_bit_precision,
        )

    builder = create(
        "qpe_circuit_builder",
        "qdk_unary",
        num_queries=num_queries,
        circuit_mapper=circuit_mapper,
        unitary_builder=AlgorithmRef("hamiltonian_unitary_builder", "sossa"),
    )
    circuit = builder.run(state_preparation=state_prep, qubit_hamiltonian=operator)[0]
    return circuit, operator


def heisenberg_queries(lambda_sos, target_precision):
    """Queries needed to resolve ``target_precision`` at the amplified band edge.

    The unary schedule wants ``num_queries + 1`` to be a power of two, so round up.
    """
    ideal = math.pi * lambda_sos / (2.0 * target_precision)
    return 2 ** math.ceil(math.log2(max(ideal, 2.0))) - 1


def logical_counts_frame(circuit, label):
    """Return the circuit's logical resource counts as a one-column frame."""
    counts = circuit.estimate().logical_counts
    return pd.DataFrame(counts.items(), columns=["Logical Estimate", label]).set_index("Logical Estimate")

> **On the sum-of-squares precondition.** `SOSSAQubitMapper` requires every per-rank sign of the
> factorization to be non-negative --- that is what makes the decomposition a genuine *sum of
> squares* and hence $H_{\mathrm{gap}} \succeq 0$. The mapper validates this condition before
> constructing the qubit operator. For a factorization of a true electron-repulsion integral tensor
> every sign should be positive, because the ERI supermatrix is a Coulomb Gram matrix and therefore
> positive semi-definite.

## Part 1 --- a stored DFTHC Hamiltonian

The first instance ships with the repository as a serialized `FactorizedHamiltonianContainer`.
It is a minimal H<sub>2</sub> problem, $N = 2$ spatial orbitals with $R = 1$ rank, $B = 2$ bases
and $C = 1$ copy, and it carries a non-zero `energy_gap`, which is the spectral-amplification
parameter: the walk is built to amplify around that gap rather than around the raw band.

In [ ]:
json_path = Path("data") / "h2_factorized_r1_b2_c1.hamiltonian.json"
h2_hamiltonian = Hamiltonian.from_json(json_path.read_text())
h2_container = h2_hamiltonian.get_container()

print(h2_hamiltonian.get_summary())

### Mapping to the SOSSA qubit operator

The mapper reads the factorization and emits the generator list together with $E_{\mathrm{SOS}}$, the
scalar that converts an $H_{\mathrm{gap}}$ eigenvalue back into a physical energy. The unitary builder
then derives $\Lambda$, half the sum of squared generator one-norms, which sets the block-encoding
normalization.

In [ ]:
h2_operator = create("qubit_mapper", "sossa").run(
    h2_hamiltonian, MajoranaMapping.jordan_wigner(2 * h2_container.get_num_orbitals())
)
h2_metadata = create("hamiltonian_unitary_builder", "sossa").run(h2_operator).get_container().metadata

print(f"Block-encoding normalization  Lambda = {h2_metadata.normalization:.6f} Hartree")
print(f"Sum-of-squares shift         E_SOS  = {h2_metadata.energy_shift:.6f} Hartree")
print(f"System qubits                       = {h2_operator.get_container().num_qubits}")

### Choosing the query schedule and sampling the circuit

Unary-iteration QPE applies the walk `num_queries` times and reads the phase off a register of
the same width. To resolve an energy $\sigma_E$ at the amplified band edge we need roughly
$\pi \Lambda / (2\sigma_E)$ queries; we round up to the next power of two minus one so the
schedule divides evenly.

The full-precision circuit is intended for resource estimation rather than local simulation. Before
estimating it, we sample a compact 7-query circuit using exact dense/direct synthesis, which avoids
the large alias-sampling and phase-gradient work registers while exercising the same SOSSA walk.

In [ ]:
from qdk.widgets import Histogram

TARGET_PRECISION = 1e-3  # Hartree ("chemical accuracy" is ~1.6e-3)
h2_queries = heisenberg_queries(h2_metadata.normalization, TARGET_PRECISION)
print(f"Queries for {TARGET_PRECISION:.0e} Ha at Lambda = {h2_metadata.normalization:.4f}: {h2_queries}")

SIMULATION_QUERIES = 7
SIMULATION_SHOTS = 256
simulation_mapper = AlgorithmRef(
    "circuit_mapper",
    "sossa",
    outer_prepare=AlgorithmRef("state_prep", "dense_pure_state"),
    inner_prepare_algorithm="direct",
    select_algorithm="direct",
)
h2_simulation_circuit, _ = sossa_unary_qpe_circuit(
    h2_hamiltonian,
    num_queries=SIMULATION_QUERIES,
    n_alpha=1,
    n_beta=1,
    circuit_mapper=simulation_mapper,
)
h2_execution = create("circuit_executor", "qdk_sparse_state_simulator", seed=20250815).run(
    h2_simulation_circuit,
    shots=SIMULATION_SHOTS,
)
h2_phase_probabilities = {
    bitstring: count / h2_execution.total_shots
    for bitstring, count in h2_execution.bitstring_counts.items()
}
display(Histogram(bar_values=h2_phase_probabilities, sort="high-to-low"))

### Logical resource estimate

Now build the full 1023-query circuit with alias-sampling PREPARE and phase-gradient SELECT, then
count its logical resources without simulating its enlarged work registers.

In [ ]:
h2_circuit, _ = sossa_unary_qpe_circuit(
    h2_hamiltonian,
    num_queries=h2_queries,
    n_alpha=1,
    n_beta=1,
)
h2_counts = logical_counts_frame(h2_circuit, "H2 SOSSA unary QPE")
display(h2_counts)

### Physical resource estimates

`qdk.qre` maps the logical circuit onto a fault-tolerant architecture and returns the
Pareto-optimal trade-offs between physical qubit count and runtime. We use a Majorana-based
architecture at a $10^{-5}$ physical error rate with the `ThreeAux` code and round-based magic
state factories, and budget 1% total error.

In [ ]:
from qdk.qre import estimate, plot_estimates
from qdk.qre.models import Majorana, RoundBasedFactory, ThreeAux

architecture = Majorana(error_rate=1e-5)
isa_query = ThreeAux.q() * RoundBasedFactory.q(use_cache=True, code_query=ThreeAux.q())

h2_results = estimate(
    h2_circuit.get_qre_application(), architecture, isa_query, max_error=0.01, name="SOSSA_H2"
)
h2_results.add_factory_summary_column()
display(h2_results.as_frame())

plot_estimates(h2_results, figsize=(6, 4), runtime_unit="ms")

## Part 2 --- stretched N<sub>2</sub> from a structure file

The second instance is built end to end. Stretching the N<sub>2</sub> bond introduces strong
multi-reference character, which is exactly the regime where a classical single-reference method
struggles and phase estimation is interesting.

The route to a SOSSA-ready Hamiltonian is:
SCF $\rightarrow$ valence space $\rightarrow$ MP2 natural-orbital localization $\rightarrow$
autoCAS-EOS active-space selection $\rightarrow$ active-space Hamiltonian $\rightarrow$
**double factorization**.

In [ ]:
from qdk_chemistry.data import Structure
from qdk_chemistry.data.symmetry import SymmetryLabel, axes

# Stretched N2 structure at 1.270025 Angstrom bond length
structure = Structure.from_xyz_file(Path("data/stretched_n2.structure.xyz"))

scf_solver = create("scf_solver")
E_hf, wfn_hf = scf_solver.run(
    structure,
    charge=0,
    spin_multiplicity=1,
    basis_or_guess="cc-pvdz",
)
print(f"Hartree-Fock energy: {E_hf:.6f} Hartree")

In [ ]:
from qdk_chemistry.utils import compute_valence_space_parameters

# Restrict to the valence space, then localize with MP2 natural orbitals
num_val_e, num_val_o = compute_valence_space_parameters(wfn_hf, charge=0)
active_space_selector = create(
    "active_space_selector",
    "qdk_valence",
    num_active_electrons=num_val_e,
    num_active_orbitals=num_val_o,
)
valence_wf = active_space_selector.run(wfn_hf)

localizer = create("orbital_localizer", "qdk_mp2_natural_orbitals")
valence_indices = valence_wf.get_orbitals().active_indices()
loc_wfn = localizer.run(
    valence_wf,
    list(valence_indices.indices(SymmetryLabel([axes.alpha()]))),
    list(valence_indices.indices(SymmetryLabel([axes.beta()]))),
)
print(f"Valence space: {num_val_e} electrons in {num_val_o} orbitals")

In [ ]:
# Selected-CI wavefunction on the localized orbitals, used to drive active-space selection
hamiltonian_constructor = create("hamiltonian_constructor")
loc_hamiltonian = hamiltonian_constructor.run(loc_wfn.get_orbitals())
num_alpha_electrons, num_beta_electrons = loc_wfn.get_active_num_electrons()

macis_mc = create(
    "multi_configuration_calculator",
    "macis_asci",
    calculate_one_rdm=True,
    calculate_two_rdm=True,
)
_, wfn_sci = macis_mc.run(loc_hamiltonian, num_alpha_electrons, num_beta_electrons)

# Entropy-based active space selection
autocas = create("active_space_selector", "qdk_autocas_eos")
autocas_wfn = autocas.run(wfn_sci)
indices = list(autocas_wfn.get_orbitals().active_indices().indices(SymmetryLabel([axes.alpha()])))
print(f"autoCAS-EOS selected {len(indices)} of {num_val_o} orbitals: indices={indices}")

In [ ]:
# Active-space Hamiltonian, plus a CASCI reference energy to benchmark against
refined_orbitals = autocas_wfn.get_orbitals()
active_hamiltonian = hamiltonian_constructor.run(refined_orbitals)

alpha_electrons, beta_electrons = autocas_wfn.get_active_num_electrons()
mc = create("multi_configuration_calculator", "macis_cas")
e_cas, wfn_cas = mc.run(active_hamiltonian, alpha_electrons, beta_electrons)
print(f"Active space CASCI energy: {e_cas:.6f} Hartree")
print(f"Active space: {alpha_electrons} alpha + {beta_electrons} beta electrons")

### Double factorization

SOSSA consumes a `FactorizedHamiltonianContainer`, so the active-space two-body integrals have to
be decomposed first. The eigen-decomposition factorizer diagonalizes the ERI supermatrix and keeps
the ranks above `truncation_threshold`.

We set the threshold to $10^{-8}$ rather than leaving it at the default $10^{-12}$ for two
reasons. It keeps $R$ --- and therefore the walk's PREPARE register and gate count --- small at a
cost far below chemical accuracy; and it discards the round-off-scale eigenvalues whose sign is
numerically meaningless, which is what keeps the sum-of-squares precondition clean.

In [ ]:
factorizer = create("double_factorizer", "eigen_decomposition")
factorizer.settings().set("truncation_threshold", 1e-8)
n2_hamiltonian = factorizer.run(active_hamiltonian)
n2_container = n2_hamiltonian.get_container()

print(n2_hamiltonian.get_summary())

Note the contrast with Part 1: the double factorizer emits `energy_gap = 0`, so this walk is *not*
spectrally amplified around a known gap. Supplying a gap estimate --- for instance from the CASCI
solve above, or from a cheaper correlated method --- is what unlocks the amplified query scaling
on a freshly factorized Hamiltonian.

In [ ]:
n2_operator = create("qubit_mapper", "sossa").run(
    n2_hamiltonian, MajoranaMapping.jordan_wigner(2 * n2_container.get_num_orbitals())
)
n2_metadata = create("hamiltonian_unitary_builder", "sossa").run(n2_operator).get_container().metadata

print(f"Block-encoding normalization  Lambda = {n2_metadata.normalization:.6f} Hartree")
print(f"Sum-of-squares shift         E_SOS  = {n2_metadata.energy_shift:.6f} Hartree")
print(f"System qubits                       = {n2_operator.get_container().num_qubits}")

### Building the N<sub>2</sub> circuit

$\Lambda$ is much larger here than for H<sub>2</sub>, so a full Heisenberg-limited schedule would
need a correspondingly larger query count. `DEMO_QUERY_CAP` keeps the notebook interactive; raise
or remove it to estimate the cost of a production-precision run.

In [ ]:
DEMO_QUERY_CAP = 255

n2_ideal_queries = heisenberg_queries(n2_metadata.normalization, TARGET_PRECISION)
n2_queries = min(n2_ideal_queries, DEMO_QUERY_CAP)
print(f"Heisenberg-limited queries for {TARGET_PRECISION:.0e} Ha: {n2_ideal_queries}")
print(f"Using {n2_queries} queries for this demo")

n2_circuit, _ = sossa_unary_qpe_circuit(
    n2_hamiltonian,
    num_queries=n2_queries,
    n_alpha=alpha_electrons,
    n_beta=beta_electrons,
)
n2_counts = logical_counts_frame(n2_circuit, "N2 SOSSA unary QPE")
display(n2_counts)

In [ ]:
n2_results = estimate(
    n2_circuit.get_qre_application(), architecture, isa_query, max_error=0.01, name="SOSSA_N2"
)
n2_results.add_factory_summary_column()
display(n2_results.as_frame())

plot_estimates(n2_results, figsize=(6, 4), runtime_unit="ms")

## Comparing the two instances

The logical counts put the two problems side by side. The dominant scaling levers are the
factorization shape $(N, R, B, C)$, which sets how much data the walk's PREPARE and SELECT must
load, and the bit precisions, which set how expensive each rotation is.

In [ ]:
comparison = pd.concat([h2_counts, n2_counts], axis=1)
display(comparison)

summary = pd.DataFrame(
    {
        "H2 (from JSON)": {
            "spatial orbitals (N)": h2_container.get_num_orbitals(),
            "ranks (R)": h2_container.get_num_ranks(),
            "lambda (Hartree)": round(h2_metadata.normalization, 6),
            "energy_gap (Hartree)": h2_container.get_energy_gap(),
            "queries": h2_queries,
        },
        "N2 (double factorized)": {
            "spatial orbitals (N)": n2_container.get_num_orbitals(),
            "ranks (R)": n2_container.get_num_ranks(),
            "lambda (Hartree)": round(n2_metadata.normalization, 6),
            "energy_gap (Hartree)": n2_container.get_energy_gap(),
            "queries": n2_queries,
        },
    }
)
display(summary)

## Table V molecular systems

To reproduce the roughly 50-spatial-orbital systems in Low *et al.* (2025), we use the optimized
DFTHC+BLISS+SA rows from Table V. Table V supplies $(N,R,B,C)$, $\lambda_{\mathrm{eff}}$, and the
published logical totals; Tables VI and VII supply the coefficient and rotation precisions used for
those rows. This distinction matters for FeMoCo: mixing its illustrative or robust-table parameters
with the optimized Table V factorization gives a different resource estimate.

The deterministic tensors below have the published shapes and synthesis precisions but are not the
molecular integral data. They are sufficient for circuit resource tracing because this implementation's
control flow depends on those dimensions and precisions. The initial determinants use each system's
published active-electron count, with the odd-electron CPD1-P450X instance assigned the minimal-spin
split $(32\alpha,31\beta)$.

For each system, the unary schedule uses

$$
p = \left\lceil \frac{\pi\lambda_{\mathrm{eff}}}{2\sigma_E} \right\rceil,
\qquad \sigma_E = 10^{-3}\ \mathrm{Ha}.
$$

Physical estimates use the same Majorana architecture, `ThreeAux` code, round-based factories, and
1% error budget as the H$_2$ and N$_2$ estimates above.

In [ ]:
import numpy as np
from qdk_chemistry.data import FactorizedHamiltonianContainer, ModelOrbitals

# Optimized DFTHC+BLISS+SA rows from Low et al. (2025), Table V.
# Bit precisions are from the corresponding rows in Tables VI and VII.
TABLE5_SYSTEMS = {
    "FeMoCo (54e, 54o)": dict(
        electrons=54,
        N=54,
        R=10,
        B=27,
        C=27,
        b_coeff=9,
        b_rot=16,
        lambda_eff=21.3674,
        paper_df_qubits=3722,
        paper_thc_qubits=2142,
        paper_sossa_qubits=1137,
        paper_sossa_toffolis=341_000_000,
        slide_toffolis=249_590_406,
        slide_qubits=1067,
        slide_uses_same_inputs=False,
    ),
    "CPD1-P450X (63e, 58o)": dict(
        electrons=63,
        N=58,
        R=9,
        B=29,
        C=14,
        b_coeff=10,
        b_rot=15,
        lambda_eff=32.7923,
        paper_df_qubits=2596,
        paper_thc_qubits=1357,
        paper_sossa_qubits=1150,
        paper_sossa_toffolis=491_000_000,
        slide_toffolis=None,
        slide_qubits=None,
        slide_uses_same_inputs=None,
    ),
    "CO2-XVIII (64e, 56o)": dict(
        electrons=64,
        N=56,
        R=5,
        B=28,
        C=28,
        b_coeff=7,
        b_rot=12,
        lambda_eff=17.0712,
        paper_df_qubits=3700,
        paper_thc_qubits=None,
        paper_sossa_qubits=924,
        paper_sossa_toffolis=205_000_000,
        slide_toffolis=139_175_084,
        slide_qubits=893,
        slide_uses_same_inputs=True,
    ),
}

SIGMA_E = 1e-3  # Hartree
MAX_ERROR = 0.01


def make_fake_factorized_hamiltonian(
    num_orbitals: int,
    num_ranks: int,
    num_bases: int,
    num_copies: int,
    seed: int = 42,
) -> FactorizedHamiltonianContainer:
    """Build a deterministic synthetic factorized Hamiltonian with the requested shape."""
    rng = np.random.default_rng(seed)
    one_body = rng.standard_normal((num_orbitals, num_orbitals))
    one_body = (one_body + one_body.T) / 2

    basis_vectors = rng.standard_normal((num_ranks, num_bases, num_orbitals))
    basis_vectors /= np.linalg.norm(basis_vectors, axis=-1, keepdims=True)

    return FactorizedHamiltonianContainer(
        0.0,
        basis_vectors.ravel(),
        rng.standard_normal((num_ranks, num_bases, num_copies)).ravel() * 0.1,
        rng.standard_normal((num_ranks, num_copies)) * 0.1,
        one_body,
        np.zeros_like(one_body),
        ModelOrbitals(num_orbitals),
    )

In [ ]:
from qdk.qre import estimate
from qdk.qre.models import Majorana, RoundBasedFactory, ThreeAux

architecture = Majorana(error_rate=1e-5)
isa_query = ThreeAux.q() * RoundBasedFactory.q(use_cache=True, code_query=ThreeAux.q())

logical_rows = []
physical_frames = []
table5_results = {}

for name, params in TABLE5_SYSTEMS.items():
    num_orbitals = int(params["N"])
    num_electrons = int(params["electrons"])
    num_queries = math.ceil(math.pi * float(params["lambda_eff"]) / (2 * SIGMA_E))
    fake_hamiltonian = Hamiltonian(
        make_fake_factorized_hamiltonian(
            num_orbitals,
            int(params["R"]),
            int(params["B"]),
            int(params["C"]),
        )
    )
    circuit, _ = sossa_unary_qpe_circuit(
        fake_hamiltonian,
        num_queries=num_queries,
        n_alpha=(num_electrons + 1) // 2,
        n_beta=num_electrons // 2,
        rotation_bit_precision=int(params["b_rot"]),
        coefficient_bit_precision=int(params["b_coeff"]),
    )

    logical_counts = dict(circuit.estimate().logical_counts)
    total_toffolis = int(logical_counts["cczCount"] + logical_counts.get("ccixCount", 0))
    logical_rows.append(
        {
            "System": name,
            "N": num_orbitals,
            "Electrons": num_electrons,
            "(R,B,C)": f"({params['R']},{params['B']},{params['C']})",
            "(b_coeff,b_rot)": f"({params['b_coeff']},{params['b_rot']})",
            "lambda_eff (Ha)": float(params["lambda_eff"]),
            "Queries": num_queries,
            "QDK logical qubits": int(logical_counts["numQubits"]),
            "QDK Toffolis": total_toffolis,
            "Paper DF qubits": int(params["paper_df_qubits"]),
            "Paper THC qubits": params["paper_thc_qubits"],
            "Paper SOSSA qubits": int(params["paper_sossa_qubits"]),
            "Paper SOSSA Toffolis": int(params["paper_sossa_toffolis"]),
        }
    )

    result = estimate(
        circuit.get_qre_application(),
        architecture,
        isa_query,
        max_error=MAX_ERROR,
        name=name,
    )
    result.add_factory_summary_column()
    table5_results[name] = result

    physical_frame = result.as_frame().copy()
    physical_frame.insert(0, "System", name)
    physical_frame.insert(1, "Pareto point", range(1, len(physical_frame) + 1))
    physical_frames.append(physical_frame.drop(columns="name", errors="ignore"))


table5_logical_estimates = pd.DataFrame(logical_rows).set_index("System")
table5_physical_estimates = pd.concat(physical_frames, ignore_index=True).set_index(
    ["System", "Pareto point"]
)

display(table5_logical_estimates)
display(table5_physical_estimates)

### Comparisons with the paper and existing slide

The Toffoli excess over the slide sits in the executable block encoding, not in
`RepeatEstimates`. CO$_2$-XVIII is the control: the slide used the same inputs for it, so its
gap is attributable to the circuit rather than to the problem. Its block originally decomposed
as

$$
10{,}216 = 2(175) + 4(1{,}371) + 2(2{,}185) + 12,
$$

for outer PREPARE, inner PREPARE, SELECT, and the inner reflection. Two QROM changes remove
avoidable work.

**The 2D SELECT-SWAP lookup folds the outer index into a single `Select`.** The lookup used to
be a `UnaryIteration` over the outer address wrapping a `Select` over the inner one. `Adjoint
AND` is free, but taking the adjoint of a whole unary iteration swaps `AND` for `Adjoint AND`
throughout, so its ladder is paid for at full price in *both* directions and the uncompute costs
as much as the compute. Addressing one flat table with the combined `(innerSelect, outer)` index
has the same forward cost and hands the uncompute to the measurement-based unlookup in
`Std.TableLookup.Select`. The flat table covers all $2^{\lceil\log_2 X_o\rceil}$ outer states
rather than only the $X_o$ valid ones, routing each unused state exactly where the unary
iteration would have, so the two paths stay identical as phase oracles. Each inner PREPARE drops
from 1,371 to 1,138 Toffolis.

**SELECT keeps its angle word loaded across the Majorana step.** `SelectImpl` wrapped the
Majorana operation in a `within`/`apply` whose *own* body was another `within`/`apply` around the
angle lookup, so the angle QROM was loaded and unloaded twice per SELECT. Passing the Majorana
step in as an action keeps the word loaded across the forward Givens chain, the Majorana
operation, and the inverse chain, leaving a single round trip. This is safe because the Majorana
operation touches only the system and spin registers, never the QROM address, so the unlookup
still addresses the entry that was loaded. Each SELECT drops from 2,185 to 1,757 Toffolis.

The optimized CO$_2$ block is therefore

$$
8{,}428 = 2(175) + 4(1{,}138) + 2(1{,}757) + 12,
$$

a 17.5% reduction. FeMoCo-54 improves by 16.5% on the same two counts, from 14,282 to 11,924
Toffolis per block excluding outer PREPARE. Because the query counts are unchanged, the totals in
the tables above scale with these block costs.

The slide's CO$_2$ total of 139,175,084 is a historical lower bound rather than an executable
clean circuit, so a residual gap to it is expected and correct. Its artifact reports 696 Toffolis
for inner PREPARE: the raw 675-Toffoli lookup plus 21 Toffolis of alias logic, with no uncompute
at all. Releasing the SELECT-SWAP workspace without the phase-repairing unlookup corrupts the
address phase, and value-only tests cannot see that omission -- which is why
`test_utils_select_swap.py` compares the swap path against the plain-select path as a *phase*
oracle. The slide's FeMoCo row also used different inputs (`b_coeff=15`, `b_rot=15`, and
$\lambda_{\mathrm{eff}}=21.4486$), and CPD1-P450X was not included.

In [ ]:
paper_comparison = table5_logical_estimates[
    [
        "QDK logical qubits",
        "Paper SOSSA qubits",
        "QDK Toffolis",
        "Paper SOSSA Toffolis",
    ]
].copy()
paper_comparison["Qubit change"] = (
    paper_comparison["QDK logical qubits"] / paper_comparison["Paper SOSSA qubits"] - 1
)
paper_comparison["Toffoli change"] = (
    paper_comparison["QDK Toffolis"] / paper_comparison["Paper SOSSA Toffolis"] - 1
)

display(
    paper_comparison.style.format(
        {
            "QDK logical qubits": "{:,}",
            "Paper SOSSA qubits": "{:,}",
            "QDK Toffolis": "{:,}",
            "Paper SOSSA Toffolis": "{:,}",
            "Qubit change": "{:+.2%}",
            "Toffoli change": "{:+.2%}",
        }
    )
)

slide_rows = []
for name, params in TABLE5_SYSTEMS.items():
    if params["slide_toffolis"] is None:
        continue

    current = table5_logical_estimates.loc[name]
    slide_rows.append(
        {
            "System": name,
            "Same inputs?": bool(params["slide_uses_same_inputs"]),
            "Current Toffolis": int(current["QDK Toffolis"]),
            "Slide Toffolis": int(params["slide_toffolis"]),
            "Toffoli change": int(current["QDK Toffolis"]) / int(params["slide_toffolis"]) - 1,
            "Current logical qubits": int(current["QDK logical qubits"]),
            "Slide logical qubits": int(params["slide_qubits"]),
            "Qubit change": int(current["QDK logical qubits"]) / int(params["slide_qubits"]) - 1,
        }
    )

slide_comparison = pd.DataFrame(slide_rows).set_index("System")
display(
    slide_comparison.style.format(
        {
            "Current Toffolis": "{:,}",
            "Slide Toffolis": "{:,}",
            "Toffoli change": "{:+.2%}",
            "Current logical qubits": "{:,}",
            "Slide logical qubits": "{:,}",
            "Qubit change": "{:+.2%}",
        }
    )
)